In [ ]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.nn as nn 
from torch.nn import CrossEntropyLoss
from torch import optim
import torch

In [49]:
# Apply transforms when creating the dataset (do not np.array the dataset)
transformer= transforms.Compose([   
    transforms.Grayscale(num_output_channels=1),  # convert to grayscale
    transforms.Resize((28,28)),   # resize to 28x28   
    transforms.ToTensor(), # convert to tensor
    transforms.Normalize((0.5,), (0.5,))   # normalization
])

In [50]:
#reading the dataset 
Train_data_path = "digit_dataset2/train"
Test_data_path = "digit_dataset2/test"
data_train = ImageFolder(Train_data_path,transform=transformer)
data_test = ImageFolder(Test_data_path,transform=transformer)
print(data_train.class_to_idx)
print(f"number of classes in Traininig set : {data_train.classes}, number of classes in Testing Set :{data_test.classes}")
print(type(data_train))
img, label = data_train[0]
print(type(img))
print(img.shape) # C, H, W


{'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9}
number of classes in Traininig set : ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'], number of classes in Testing Set :['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
<class 'torchvision.datasets.folder.ImageFolder'>
<class 'torch.Tensor'>
torch.Size([1, 28, 28])


In [51]:
#wrapping the data into the DataLoader
train_loader = DataLoader(data_train, batch_size=32, shuffle=True)
test_loader = DataLoader(data_test, batch_size=32, shuffle=False)

In [52]:
#Building the Neural Network Model (fully Connected)
FCN = nn.Sequential(
    nn.Flatten(),  # flatten input image
    nn.Linear(784,128),  # input layer to hidden layer
    nn.ReLU(), # activation function
    nn.Linear(128,64),  # hidden layer to hidden layer
    nn.ReLU(), # activation function
    nn.Dropout(0.5), # dropout layer for regularization
    nn.Linear(64,10),   # hidden layer to output layer
   
 )
    

In [56]:
#training the model 
loss_function = CrossEntropyLoss() # loss function for multi-class classification
optimizer = optim.SGD(FCN.parameters(),lr = 0.03) # Stochastic Gradient Descent optimizer


In [ ]:
#Setting the model to Training mode
FCN.train()
epochs = 5
for epoch in range(epochs):
    for images, labels in train_loader:
        # 1. Forward pass
        outputs = FCN(images)
        
        # 2. Compute loss
        loss = loss_fn(outputs, labels)
        
        # 3. Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # 4. Update weights
        optimizer.step()
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.2f}")

In [ ]:
#evaluating mode 
FCN.eval()  # disables dropout
with torch.no_grad():  # no need to compute gradients
    correct = 0
    total = 0
    for images, labels in test_loader:
        outputs = FCN(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Test Accuracy: {100 * correct / total:.2f}%")
